In [1]:
from pathlib import Path
import json

import cv2
cv2.setNumThreads(0)  

import torch
import pandas as pd
import numpy as np

from ultralytics import YOLO

In [2]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Selected device:", DEVICE)
if DEVICE == 0:
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: 0
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
PROJECT_ROOT = Path("..").resolve()
DATASET_VERSION = "v4"
STAGE1_NAME = "stage1"
STAGE2_NAME = "stage2"

STAGE1_BEST_PATH = (
    PROJECT_ROOT
    / "runs"
    / DATASET_VERSION
    / "segmentation"
    / STAGE1_NAME
    / "weights"
    / "best.pt"
)

assert STAGE1_BEST_PATH.exists(), f"Checkpoint stage 1 not found :\n{STAGE1_BEST_PATH}"

model = YOLO(str(STAGE1_BEST_PATH))
print("Warm-start from checkpoint stage 1:")
print(STAGE1_BEST_PATH)

Warm-start from checkpoint stage 1:
D:\PREP_INTERN\nutrivision_pro\runs\v4\segmentation\stage1\weights\best.pt


In [4]:
print("=== STATUS BEFORE STAGE 2 ===")
for i, layer in enumerate(model.model.model):
    params = list(layer.parameters())
    status = "NO PARAMS" if not params else ("TRAINABLE" if any(p.requires_grad for p in params) else "FROZEN")
    print(f"{i:02d} | {layer.__class__.__name__:12s} | {status}")

=== STATUS BEFORE STAGE 2 ===
00 | Conv         | FROZEN
01 | Conv         | FROZEN
02 | C3k2         | FROZEN
03 | Conv         | FROZEN
04 | C3k2         | FROZEN
05 | Conv         | FROZEN
06 | C3k2         | FROZEN
07 | Conv         | FROZEN
08 | C3k2         | FROZEN
09 | SPPF         | FROZEN
10 | C2PSA        | FROZEN
11 | Upsample     | NO PARAMS
12 | Concat       | NO PARAMS
13 | C3k2         | FROZEN
14 | Upsample     | NO PARAMS
15 | Concat       | NO PARAMS
16 | C3k2         | FROZEN
17 | Conv         | FROZEN
18 | Concat       | NO PARAMS
19 | C3k2         | FROZEN
20 | Conv         | FROZEN
21 | Concat       | NO PARAMS
22 | C3k2         | FROZEN
23 | Segment      | FROZEN


In [5]:
DATASET_DIR = PROJECT_ROOT / "datasets" / DATASET_VERSION
DATA_YAML = DATASET_DIR / "data.yaml"

RUNS_DIR = PROJECT_ROOT / "runs"
EXPERIMENT_DIR = RUNS_DIR / DATASET_VERSION / "segmentation" / STAGE2_NAME
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
STAGE2_CONFIG = {
    "data": str(DATA_YAML),
    "epochs": 30,
    "imgsz": 640,
    "batch": 8,
    "device": DEVICE,
    "freeze": 0,             # SEMUA layer dicairkan -- inti dari stage 2
    "workers": 2,
    "save_period": 5,

    "optimizer": "AdamW",
    "lr0": 0.0001,            # 10x lebih kecil dari stage 1 (0.001) -- cegah catastrophic forgetting
    "lrf": 0.01,
    "weight_decay": 0.0005,

    "warmup_epochs": 1,        # lebih pendek: backbone sudah "hangat" dari stage 1, bukan random lagi
    "patience": 15,

    # Augmentasi -- disamakan dengan stage 1, supaya perbandingan hasil adil
    # (kalau augmentasi beda, kita tidak akan tahu apakah perubahan performa
    # berasal dari unfreeze atau dari augmentasi yang beda)
    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
    "degrees": 5.0, "translate": 0.1, "scale": 0.5, "fliplr": 0.5,
    "copy_paste": 0.4,
    "copy_paste_mode": "mixup",
    "erasing": 0.1,
    "mask_ratio": 2,

    "amp": True,
    "seed": 42,
}

print(json.dumps(STAGE2_CONFIG, indent=2))

{
  "data": "D:\\PREP_INTERN\\nutrivision_pro\\datasets\\v4\\data.yaml",
  "epochs": 30,
  "imgsz": 640,
  "batch": 8,
  "device": 0,
  "freeze": 0,
  "workers": 2,
  "save_period": 5,
  "optimizer": "AdamW",
  "lr0": 0.0001,
  "lrf": 0.01,
  "weight_decay": 0.0005,
  "warmup_epochs": 1,
  "patience": 15,
  "hsv_h": 0.015,
  "hsv_s": 0.7,
  "hsv_v": 0.4,
  "degrees": 5.0,
  "translate": 0.1,
  "scale": 0.5,
  "fliplr": 0.5,
  "copy_paste": 0.4,
  "copy_paste_mode": "mixup",
  "erasing": 0.1,
  "mask_ratio": 2,
  "amp": true,
  "seed": 42
}


In [9]:

LAST_CKPT = EXPERIMENT_DIR / "weights" / "last.pt"
resume_arg = str(LAST_CKPT) if LAST_CKPT.exists() else False

stage2_results = model.train(
    **STAGE2_CONFIG,
    resume=resume_arg,
    project=str(EXPERIMENT_DIR.parent),
    name=EXPERIMENT_DIR.name,
    exist_ok=True,
    plots=True,
    verbose=True,
)
RUN_DIR = Path(model.trainer.save_dir)
print(f"\nRun saved in : {RUN_DIR}")

New https://pypi.org/project/ultralytics/8.4.152 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.147  Python-3.13.5 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
The CUDA driver logged these messages, which may provide useful details:
Sticky error detected
Returning 2 (CUDA_ERROR_OUT_OF_MEMORY) from cuStreamGetCaptureInfo_v3



In [ ]:
print("=== VERIFICATION STAGE 2 ===")
trained = YOLO(RUN_DIR / "weights" / "last.pt")
for i, layer in enumerate(trained.model.model):
    params = list(layer.parameters())
    status = "NO PARAMS" if not params else ("TRAINABLE" if any(p.requires_grad for p in params) else "FROZEN")
    print(f"{i:02d} | {layer.__class__.__name__:12s} | {status}")

=== VERIFICATION STAGE 2 ===
00 | Conv         | FROZEN
01 | Conv         | FROZEN
02 | C3k2         | FROZEN
03 | Conv         | FROZEN
04 | C3k2         | FROZEN
05 | Conv         | FROZEN
06 | C3k2         | FROZEN
07 | Conv         | FROZEN
08 | C3k2         | FROZEN
09 | SPPF         | FROZEN
10 | C2PSA        | FROZEN
11 | Upsample     | NO PARAMS
12 | Concat       | NO PARAMS
13 | C3k2         | FROZEN
14 | Upsample     | NO PARAMS
15 | Concat       | NO PARAMS
16 | C3k2         | FROZEN
17 | Conv         | FROZEN
18 | Concat       | NO PARAMS
19 | C3k2         | FROZEN
20 | Conv         | FROZEN
21 | Concat       | NO PARAMS
22 | C3k2         | FROZEN
23 | Segment      | FROZEN


In [ ]:
BEST_MODEL_PATH = EXPERIMENT_DIR / "weights" / "best.pt"
assert BEST_MODEL_PATH.exists(), f"Best model not found:\n{BEST_MODEL_PATH}"

model = YOLO(str(BEST_MODEL_PATH))
print("Best model loaded.")
print("Model path :", BEST_MODEL_PATH)
print("Dataset    :", DATASET_VERSION)
print("Stage      :", STAGE2_NAME)


Best model loaded.
Model path : D:\PREP_INTERN\nutrivision_pro\runs\v3\segmentation\stage2\weights\best.pt
Dataset    : v3
Stage      : stage2


In [ ]:
results = model.val(
    data=DATA_YAML,
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    plots=True,
    verbose=True,
    augment=True,
)

Ultralytics 8.4.147  Python-3.13.5 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11n-seg summary (fused): 113 layers, 2,837,493 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 188.244.9 MB/s, size: 64.6 KB)
val: Scanning D:\PREP_INTERN\nutrivision_pro\datasets\v3\valid\labels.cache... 56 images, 0 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 56/56 7.1Mit/s 0.0s
val: D:\PREP_INTERN\nutrivision_pro\datasets\v3\valid\images\makanan_222_jpg.rf.e97d28e0a26b2bf386fa8d0a46d10625.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: D:\PREP_INTERN\nutrivision_pro\datasets\v3\valid\images\makanan_242_jpg.rf.5db86ae95fc8fb7abddb456572d830e8.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: D:\PREP_INTERN\nutrivision_pro\datasets\v3\valid\images\makanan_64_jpg.rf.7f7726b64ddd480e4d5f20266a72b77e.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: D:\PREP_INT

In [ ]:
class_names = model.names
class_indices = results.seg.ap_class_index

class_metrics = pd.DataFrame({
    "class_id": class_indices,
    "class_name": [class_names[int(i)] for i in class_indices],
    "precision": results.seg.p,
    "recall": results.seg.r,
    "mAP50": results.seg.ap50,
    "mAP50-95": results.seg.ap,
})

class_metrics["f1"] = (
    2 * class_metrics["precision"] * class_metrics["recall"]
    / (class_metrics["precision"] + class_metrics["recall"]).replace(0, np.nan)
)

class_metrics = class_metrics.round(4)
class_metrics

,class_id,class_name,precision,recall,mAP50,mAP50-95,f1
0,0,beef,0.3515,0.2273,0.2012,0.1783,0.2760
1,1,chicken,0.4427,0.3782,0.3241,0.2089,0.4079
2,2,egg,0.1990,0.5928,0.3641,0.3060,0.2980
3,3,fish,0.3617,0.4752,0.3383,0.1893,0.4108
4,4,fruit,0.7934,0.7941,0.8681,0.6569,0.7937
5,5,noodles,0.4107,0.6000,0.4004,0.2310,0.4876
6,6,other_carbs,0.0000,0.0000,0.0234,0.0156,NaN
7,7,pork,0.5238,0.4783,0.4433,0.2824,0.5000
8,8,rice,0.6634,0.6585,0.6807,0.5705,0.6610
9,9,sambal,0.5237,1.0000,0.8630,0.7767,0.6874


In [ ]:
# Cari threshold confidence yang memaksimalkan F1 per kelas dari kurva PR,
# bukan pakai satu angka default 0.25 untuk semua kelas
f1_curve = results.seg.curves_results[1]   # index 1 = "F1-Confidence(M)"
conf_values, f1_matrix = f1_curve[0], f1_curve[1]

optimal_thresholds = {}
for i, class_id in enumerate(class_indices):
    class_name = class_names[int(class_id)]
    best_idx = f1_matrix[i].argmax()
    optimal_thresholds[class_name] = round(float(conf_values[best_idx]), 3)

print("=== Optimal confidence threshold per kelas ===")
for name, thr in optimal_thresholds.items():
    print(f"{name:15s} : {thr}")

THRESHOLD_JSON = EXPERIMENT_DIR / "optimal_class_thresholds.json"
with open(THRESHOLD_JSON, "w") as f:
    json.dump(optimal_thresholds, f, indent=2)
print(f"\nDisimpan ke: {THRESHOLD_JSON}")

In [ ]:
CSV_PATH = (
    PROJECT_ROOT
    / "runs"
    / DATASET_VERSION
    / "segmentation"
    / STAGE2_NAME
    / "per_class_segmentation_metrics.csv"
)
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
class_metrics.to_csv(CSV_PATH, index=False)

print("CSV saved to:", CSV_PATH.resolve())

CSV saved to: D:\PREP_INTERN\nutrivision_pro\runs\v3\segmentation\stage2\per_class_segmentation_metrics.csv


In [ ]:
def evaluate_per_image_class_presence(model, images_dir, labels_dir, conf_thresholds, default_conf=0.25):
    class_names_list = list(model.names.values())
    name_to_id = {v: k for k, v in model.names.items()}
    results_log = []

    for image_path in sorted(images_dir.glob("*.jpg")):
        label_path = labels_dir / f"{image_path.stem}.txt"
        gt_classes = set()
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                if line.strip():
                    gt_classes.add(int(line.split()[0]))

        # prediksi dengan confidence PALING RENDAH dulu, lalu filter manual per kelas
        # pakai threshold optimal masing-masing (bukan satu conf global)
        pred = model.predict(image_path, conf=0.05, verbose=False)[0]
        pred_classes = set()
        if pred.boxes is not None:
            for cls_idx, conf in zip(pred.boxes.cls, pred.boxes.conf):
                cname = class_names_list[int(cls_idx)]
                threshold = conf_thresholds.get(cname, default_conf)
                if float(conf) >= threshold:
                    pred_classes.add(int(cls_idx))

        false_negative = gt_classes - pred_classes
        false_positive = pred_classes - gt_classes

        results_log.append({
            "image": image_path.name,
            "gt_classes": len(gt_classes),
            "correctly_identified": len(gt_classes & pred_classes),
            "missed_classes": len(false_negative),
            "spurious_classes": len(false_positive),
            "image_perfect": len(false_negative) == 0 and len(false_positive) == 0,
        })

    df = pd.DataFrame(results_log)
    perfect_pct = df["image_perfect"].mean() * 100
    recall_pct = (df["correctly_identified"].sum() / (df["gt_classes"].sum() + 1e-9)) * 100

    print(f"Gambar dengan SEMUA kelas benar (tidak ada miss/nyasar): "
          f"{df['image_perfect'].sum()} / {len(df)} ({perfect_pct:.1f}%)")
    print(f"Rata-rata kelas ter-recall per gambar: {recall_pct:.1f}%")
    print(f"Rata-rata spurious (kelas nyasar) per gambar: {df['spurious_classes'].mean():.2f}")
    return df


per_image_df = evaluate_per_image_class_presence(
    model,
    images_dir=DATASET_DIR / "valid" / "images",
    labels_dir=DATASET_DIR / "valid" / "labels",
    conf_thresholds=optimal_thresholds,
)

PER_IMAGE_CSV = EXPERIMENT_DIR / "per_image_class_presence.csv"
per_image_df.to_csv(PER_IMAGE_CSV, index=False)
print(f"\nDisimpan ke: {PER_IMAGE_CSV}")

## optuna


In [ ]:
# # pip install -U "ray[tune]" optuna   (jalankan sekali di terminal, bukan di notebook)

# TUNE_CONFIG = dict(
#     data=str(DATA_YAML),
#     epochs=12,          # SINGKAT per trial -- ini eksplorasi, bukan training penuh
#     iterations=20,       # jumlah trial total
#     optimizer="AdamW",
#     search_alg="optuna",  # Bayesian optimization, bukan genetic evolution bawaan
#     use_ray=True,
#     plots=False,
#     save=False,
#     workers=2,
#     device=DEVICE,
#     project=str(RUNS_DIR / DATASET_VERSION / "segmentation" / "optuna_search"),
# )

# # warm-start dari checkpoint stage 2 terbaik -- bukan dari yolo11n-seg.pt COCO polos,
# # supaya pencarian hyperparameter relevan dengan kondisi model yang SUDAH domain-adapted
# tune_model = YOLO(str(EXPERIMENT_DIR / "weights" / "best.pt"))
# tune_result = tune_model.tune(**TUNE_CONFIG)

In [ ]:
# best_hyperparameters = tune_result.get_best_result().config
# print("=== Kandidat hyperparameter terbaik dari Optuna ===")
# for k, v in best_hyperparameters.items():
#     print(f"{k}: {v}")

# print("\nBandingkan manual dengan STAGE2_CONFIG saat ini sebelum diadopsi:")
# for k in ["lr0", "lrf", "weight_decay", "box", "cls", "warmup_epochs"]:
#     if k in best_hyperparameters:
#         print(f"  {k}: Optuna={best_hyperparameters[k]:.6f} | Current={STAGE2_CONFIG.get(k, 'default')}")